In [ ]:
# notebooks/task2_chunking_embedding.ipynb
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Task 2: Text Chunking, Embedding, and Vector Store Indexing\n",
    "\n",
    "This notebook demonstrates the complete pipeline for:\n",
    "1. Creating stratified samples of complaint data\n",
    "2. Chunking complaint narratives\n",
    "3. Generating embeddings\n",
    "4. Building vector stores (FAISS/ChromaDB)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import sys\n",
    "import os\n",
    "sys.path.append('../src')\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from pathlib import Path\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set display options\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.max_colwidth', 200)\n",
    "\n",
    "print(\"Libraries imported successfully!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import our VectorStoreBuilder\n",
    "from task2_chunking_embedding import VectorStoreBuilder\n",
    "\n",
    "# Initialize the builder\n",
    "builder = VectorStoreBuilder(\n",
    "    embedding_model_name='all-MiniLM-L6-v2',\n",
    "    chunk_size=500,\n",
    "    chunk_overlap=50,\n",
    "    sample_size=15000,\n",
    "    random_state=42\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load and Explore Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load the cleaned data from Task 1\n",
    "data_path = '../data/processed/filtered_complaints.csv'\n",
    "data = builder.load_data(data_path)\n",
    "\n",
    "# Display data info\n",
    "print(f\"Data shape: {data.shape}\")\n",
    "print(f\"\\nFirst few rows:\")\n",
    "data.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze text length distribution\n",
    "if 'cleaned_word_count' in data.columns:\n",
    "    fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "    \n",
    "    # Histogram\n",
    "    axes[0].hist(data['cleaned_word_count'], bins=50, edgecolor='black', alpha=0.7)\n",
    "    axes[0].set_xlabel('Word Count')\n",
    "    axes[0].set_ylabel('Frequency')\n",
    "    axes[0].set_title('Distribution of Word Counts')\n",
    "    axes[0].axvline(data['cleaned_word_count'].median(), color='red', linestyle='--', \n",
    "                   label=f'Median: {data[\"cleaned_word_count\"].median():.0f}')\n",
    "    axes[0].legend()\n",
    "    \n",
    "    # Box plot by product\n",
    "    product_data = []\n",
    "    labels = []\n",
    "    for product in data['product_category'].unique():\n",
    "        product_words = data[data['product_category'] == product]['cleaned_word_count']\n",
    "        if len(product_words) > 0:\n",
    "            product_data.append(product_words)\n",
    "            labels.append(product)\n",
    "    \n",
    "    axes[1].boxplot(product_data, labels=labels)\n",
    "    axes[1].set_ylabel('Word Count')\n",
    "    axes[1].set_title('Word Count by Product Category')\n",
    "    axes[1].tick_params(axis='x', rotation=45)\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Create Stratified Sample"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create stratified sample\n",
    "sampled_data = builder.create_stratified_sample()\n",
    "\n",
    "# Visualize sample distribution\n",
    "fig, ax = plt.subplots(figsize=(10, 6))\n",
    "product_counts = sampled_data['product_category'].value_counts()\n",
    "colors = plt.cm.Set3(np.linspace(0, 1, len(product_counts)))\n",
    "\n",
    "bars = ax.bar(product_counts.index, product_counts.values, color=colors)\n",
    "ax.set_xlabel('Product Category')\n",
    "ax.set_ylabel('Number of Complaints')\n",
    "ax.set_title('Stratified Sample Distribution')\n",
    "ax.tick_params(axis='x', rotation=45)\n",
    "\n",
    "# Add value labels\n",
    "for bar, value in zip(bars, product_counts.values):\n",
    "    height = bar.get_height()\n",
    "    ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,\n",
    "            f'{value:,}', ha='center', va='bottom', fontsize=10)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Initialize Components"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize embedding model and text splitter\n",
    "builder.initialize_components()\n",
    "\n",
    "# Test the embedding model\n",
    "test_texts = [\"credit card fraud complaint\", \"loan payment issue\", \"savings account problem\"]\n",
    "test_embeddings = builder.embedding_model.encode(test_texts)\n",
    "print(f\"Test embeddings shape: {test_embeddings.shape}\")\n",
    "print(f\"Embedding dimension: {builder.embedding_dim}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Chunk Complaint Narratives"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Process complaints (chunking)\n",
    "chunks, metadata = builder.process_complaints(sampled_data)\n",
    "\n",
    "# Analyze chunk statistics\n",
    "chunk_lengths = [len(chunk) for chunk in chunks]\n",
    "chunks_per_complaint = [meta['total_chunks'] for meta in metadata if 'total_chunks' in meta]\n",
    "\n",
    "fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n",
    "\n",
    "# Chunk length distribution\n",
    "axes[0].hist(chunk_lengths, bins=50, edgecolor='black', alpha=0.7)\n",
    "axes[0].set_xlabel('Chunk Length (characters)')\n",
    "axes[0].set_ylabel('Frequency')\n",
    "axes[0].set_title(f'Chunk Length Distribution\\nMean: {np.mean(chunk_lengths):.0f}, Median: {np.median(chunk_lengths):.0f}')\n",
    "\n",
    "# Chunks per complaint\n",
    "if chunks_per_complaint:\n",
    "    axes[1].hist(chunks_per_complaint, bins=30, edgecolor='black', alpha=0.7, color='green')\n",
    "    axes[1].set_xlabel('Chunks per Complaint')\n",
    "    axes[1].set_ylabel('Frequency')\n",
    "    axes[1].set_title(f'Chunks per Complaint\\nMean: {np.mean(chunks_per_complaint):.1f}')\n",
    "\n",
    "# Chunk length by product\n",
    "chunk_df = pd.DataFrame({\n",
    "    'chunk_length': chunk_lengths,\n",
    "    'product': [meta.get('product_category', 'Unknown') for meta in metadata]\n",
    "})\n",
    "\n",
    "product_avg = chunk_df.groupby('product')['chunk_length'].mean().sort_values()\n",
    "axes[2].barh(range(len(product_avg)), product_avg.values, color=plt.cm.Set3(np.linspace(0, 1, len(product_avg))))\n",
    "axes[2].set_yticks(range(len(product_avg)))\n",
    "axes[2].set_yticklabels(product_avg.index)\n",
    "axes[2].set_xlabel('Average Chunk Length')\n",
    "axes[2].set_title('Average Chunk Length by Product')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Show sample chunks\n",
    "print(\"\\nSample chunks:\")\n",
    "for i in range(3):\n",
    "    print(f\"\\n--- Chunk {i+1} ---\")\n",
    "    print(f\"Product: {metadata[i].get('product_category', 'Unknown')}\")\n",
    "    print(f\"Length: {len(chunks[i])} characters\")\n",
    "    print(f\"Text: {chunks[i][:200]}...\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Generate Embeddings"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate embeddings\n",
    "embeddings = builder.generate_embeddings()\n",
    "\n",
    "# Analyze embedding statistics\n",
    "print(f\"Embeddings shape: {embeddings.shape}\")\n",
    "print(f\"Embeddings dtype: {embeddings.dtype}\")\n",
    "print(f\"\\nEmbedding statistics:\")\n",
    "print(f\"  Min: {embeddings.min():.4f}\")\n",
    "print(f\"  Max: {embeddings.max():.4f}\")\n",
    "print(f\"  Mean: {embeddings.mean():.4f}\")\n",
    "print(f\"  Std: {embeddings.std():.4f}\")\n",
    "\n",
    "# Visualize embedding distribution\n",
    "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n",
    "\n",
    "# Histogram of all embedding values\n",
    "axes[0].hist(embeddings.flatten(), bins=100, edgecolor='black', alpha=0.7)\n",
    "axes[0].set_xlabel('Embedding Value')\n",
    "axes[0].set_ylabel('Frequency')\n",
    "axes[0].set_title('Distribution of Embedding Values')\n",
    "\n",
    # Mean embedding by dimension\n",
    "mean_by_dim = embeddings.mean(axis=0)\n",
    "axes[1].plot(range(len(mean_by_dim)), mean_by_dim, linewidth=1)\n",
    "axes[1].fill_between(range(len(mean_by_dim)), \n",
    "                     mean_by_dim - embeddings.std(axis=0),\n",
    "                     mean_by_dim + embeddings.std(axis=0),\n",
    "                     alpha=0.3)\n",
    "axes[1].set_xlabel('Dimension')\n",
    "axes[1].set_ylabel('Mean Value')\n",
    "axes[1].set_title('Mean Embedding by Dimension')\n",
    "axes[1].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Build Vector Store (ChromaDB)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Build ChromaDB vector store\n",
    "chroma_collection = builder.build_chromadb_store(embeddings)\n",
    "\n",
    "# Test retrieval\n",
    "print(\"\\nTesting retrieval with sample queries:\")\n",
    "\n",
    "test_queries = [\n",
    "    \"credit card unauthorized charges\",\n",
    "    \"loan payment delayed\",\n",
    "    \"savings account withdrawal denied\",\n",
    "    \"money transfer failed\"\n",
    "]\n",
    "\n",
    "for query in test_queries:\n",
    "    print(f\"\\nQuery: '{query}'\")\n",
    "    \n",
    "    # Generate query embedding\n",
    "    query_embedding = builder.embedding_model.encode([query])[0]\n",
    "    \n",
    "    # Search\n",
    "    results = chroma_collection.query(\n",
    "        query_embeddings=[query_embedding.tolist()],\n",
    "        n_results=3,\n",
    "        include=['documents', 'metadatas', 'distances']\n",
    "    )\n",
    "    \n",
    "    if results['documents'] and results['documents'][0]:\n",
    "        for i, (doc, meta, dist) in enumerate(zip(\n",
    "            results['documents'][0],\n",
    "            results['metadatas'][0],\n",
    "            results['distances'][0]\n",
    "        )):\n",
    "            product = meta.get('product_category', 'Unknown')\n",
    "            print(f\"  Match {i+1}: Product={product}, Distance={dist:.4f}\")\n",
    "            print(f\"     Preview: {doc[:100]}...\")\n",
    "    else:\n",
    "        print(\"  No results found\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Build Vector Store (FAISS) - Alternative"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Optional: Build FAISS index\n",
    "faiss_index = builder.build_faiss_index(embeddings)\n",
    "\n",
    "# Test FAISS retrieval\n",
    "print(\"\\nTesting FAISS retrieval:\")\n",
    "\n",
    "# Test with sample query\n",
    "test_query = \"bank account frozen\"\n",
    "query_embedding = builder.embedding_model.encode([test_query])[0]\n",
    "\n",
    # Normalize for FAISS\n",
    "query_embedding_normalized = query_embedding.copy()\n",
    "query_embedding_normalized = query_embedding_normalized.reshape(1, -1)\n",
    "faiss.normalize_L2(query_embedding_normalized)\n",
    "\n",
    "# Search\n",
    "k = 5\n",
    "distances, indices = faiss_index.search(query_embedding_normalized, k)\n",
    "\n",
    "print(f\"Query: '{test_query}'\")\n",
    "print(f\"Top {k} matches:\")\n",
    "for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):\n",
    "    if idx != -1:\n",
    "        product = builder.metadata[idx].get('product_category', 'Unknown')\n",
    "        chunk_text = builder.chunks[idx][:100]\n",
    "        print(f\"  Match {i+1}: Index={idx}, Product={product}, Score={dist:.4f}\")\n",
    "        print(f\"     Preview: {chunk_text}...\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Save Output in Parquet Format"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save embeddings and metadata in Parquet format\n",
    "parquet_df = builder.save_parquet_output()\n",
    "\n",
    "# Display Parquet data info\n",
    "print(f\"Parquet DataFrame shape: {parquet_df.shape}\")\n",
    "print(f\"\\nColumns: {list(parquet_df.columns)}\")\n",
    "print(f\"\\nFirst few rows:\")\n",
    "parquet_df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Performance Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze retrieval performance\n",
    "print(\"Performance Analysis:\")\n",
    "print(\"=\"*50)\n",
    "\n",
    "# Chunk statistics\n",
    "print(f\"\\n1. Chunking Statistics:\")\n",
    "print(f\"   Total complaints: {len(sampled_data)}\")\n",
    "print(f\"   Total chunks: {len(builder.chunks)}\")\n",
    "print(f\"   Chunks per complaint: {len(builder.chunks) / len(sampled_data):.2f}\")\n",
    "print(f\"   Average chunk length: {np.mean([len(c) for c in builder.chunks]):.0f} chars\")\n",
    "\n",
    "# Embedding statistics\n",
    "print(f\"\\n2. Embedding Statistics:\")\n",
    "print(f\"   Embedding dimension: {builder.embedding_dim}\")\n",
    "print(f\"   Total embeddings: {len(embeddings)}\")\n",
    "print(f\"   Memory usage: {embeddings.nbytes / 1024 / 1024:.2f} MB\")\n",
    "\n",
    "# Vector store statistics\n",
    "print(f\"\\n3. Vector Store Statistics:\")\n",
    "print(f\"   Type: {'ChromaDB' if hasattr(builder, 'chroma_collection') else 'FAISS'}\")\n",
    "print(f\"   Index size: {len(builder.chunks)} vectors\")\n",
    "\n",
    "# Sample similarity analysis\n",
    "print(f\"\\n4. Similarity Analysis (sample):\")\n",
    "\n",
    "# Calculate similarity between random chunks\n",
    "np.random.seed(42)\n",
    "sample_indices = np.random.choice(len(embeddings), 100, replace=False)\n",
    "sample_embeddings = embeddings[sample_indices]\n",
    "\n",
    "# Calculate cosine similarities\n",
    "from sklearn.metrics.pairwise import cosine_similarity\n",
    "similarity_matrix = cosine_similarity(sample_embeddings)\n",
    "\n",
    "# Get off-diagonal similarities (comparisons between different chunks)\n",
    "off_diagonal = similarity_matrix[~np.eye(similarity_matrix.shape[0], dtype=bool)]\n",
    "\n",
    "print(f\"   Average similarity between random chunks: {off_diagonal.mean():.4f}\")\n",
    "print(f\"   Std similarity: {off_diagonal.std():.4f}\")\n",
    "print(f\"   Min similarity: {off_diagonal.min():.4f}\")\n",
    "print(f\"   Max similarity: {off_diagonal.max():.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Conclusion\n",
    "\n",
    "Task 2 has been completed successfully! We have:\n",
    "1. **Created a stratified sample** of complaint data\n",
    "2. **Chunked narratives** into manageable pieces\n",
    "3. **Generated embeddings** using sentence-transformers\n",
    "4. **Built vector stores** (ChromaDB and FAISS)\n",
    "5. **Saved output** in multiple formats\n",
    "\n",
    "The vector store is now ready for Task 3: Building the RAG pipeline."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}